In [25]:
import pandas as pd
import numpy as np
import os
import sys
from datetime import datetime
import plotly.express as px

Set variables

Import Data

In [ ]:

current_path = os.getcwd()
gold_path = os.path.abspath(os.path.join(current_path, "bucket", "gold"))

def find_latest_partition(base_path):
    partitions = []
    for root, dirs, files in os.walk(base_path):
        if "year=" in root and "month=" in root and "day=" in root:
            parts = root.split(os.sep)
            year = int([p.split('=')[1] for p in parts if p.startswith('year=')][0])
            month = int([p.split('=')[1] for p in parts if p.startswith('month=')][0])
            day = int([p.split('=')[1] for p in parts if p.startswith('day=')][0])
            partitions.append((year, month, day, root))
    
    if not partitions:
        raise FileNotFoundError(f"No partitions found in {base_path}")

    partitions.sort()
    return partitions[-1][-1]  # Return the latest partition path

# Find latest gold partition
latest_gold_partition = find_latest_partition(gold_path)

print(f"✅ Latest partition found: {latest_gold_partition}")

# Now load the Parquet files
df_appointments = pd.read_parquet(os.path.join(latest_gold_partition, "appointments.parquet"))
df_patients = pd.read_parquet(os.path.join(latest_gold_partition, "patients.parquet"))
df_prescriptions = pd.read_parquet(os.path.join(latest_gold_partition, "prescriptions.parquet"))
df_providers = pd.read_parquet(os.path.join(latest_gold_partition, "providers.parquet"))

print("✅ Data successfully loaded from latest gold partition!")


✅ Latest partition found: /Users/jvclark/www/portrailcare/case/portrait-data-engineer-test/app/bucket/gold/year=2025/month=04/day=28
✅ Data successfully loaded from latest gold partition!


In [29]:
# What is the distribution of patients across age groups?
df_patients['age_group'].value_counts()

age_group
71+        15
31-50      14
51-70      12
19-30       8
Unknown     5
0-18        1
Name: count, dtype: int64

In [30]:
#How does the appointment frequency vary by patient type? 
df1 = df_appointments.merge(
    df_patients,
    on='patient_id',
    how='left'
)

df1 = df1[df1['days_since_last_appointment'] >= 0]


analysis = df1.groupby('patient_type')['days_since_last_appointment'].agg(['count', 'mean', 'median', 'std']).round(2)
analysis

/var/folders/qs/t5dq2fb92vx3_q20m013swcm0000gn/T/ipykernel_48917/191632202.py:11: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,count,mean,median,std
patient_type,,,,
New,2,9.50,9.5,2.12
Regular,37,25.65,17.0,21.42
Long-term,18,23.11,10.0,25.77
Unknown,0,NaN,NaN,NaN


In [31]:
# What are the most common appointment types by age group?
appointment_counts = df1.groupby(['age_group', 'appointment_type']).size().reset_index(name='count')

max_counts = appointment_counts.groupby('age_group')['count'].transform('max')
top_appointment_counts = appointment_counts[appointment_counts['count'] == max_counts]
top_appointment_counts = top_appointment_counts.sort_values(['age_group', 'appointment_type']).reset_index(drop=True)

fig = px.bar(
    appointment_counts,
    x='age_group',
    y='count',
    color='appointment_type',
    barmode='group',
    title='Tipos de Consulta Mais Comuns por Faixa Etária',
    labels={
        'age_group': 'Faixa Etária',
        'count': 'Número de Consultas',
        'appointment_type': 'Tipo de Consulta'
    },
    width=900,
    height=500
)

fig.show()


top_appointment_counts

,age_group,appointment_type,count
0,0-18,Consultation,1
1,19-30,Checkup,4
2,19-30,Consultation,4
3,19-30,Emergency,4
4,31-50,Consultation,5
5,31-50,Emergency,5
6,51-70,Checkup,7
7,51-70,Emergency,7
8,71+,Checkup,9


Are there specific days of the week with higher emergency visits?

In [32]:
#Are there specific days of the week with higher emergency visits?
df_appointments[df_appointments['appointment_type'] == 'Emergency'].groupby('day_of_week')["appointment_id"].count().reset_index(name='count').sort_values(by='count', ascending=False)

,day_of_week,count
0,Friday,9
1,Monday,6
2,Saturday,6
4,Thursday,4
3,Sunday,3
5,Tuesday,3
6,Wednesday,2


In [33]:
df2 

,prescription_id,patient_id,medication_name,prescription_date,extraction_timestamp_x,updated_at_x,prescription_frequency,avg_prescription_frequency,prescription_repeats,name,age,gender,registration_date,extraction_timestamp_y,updated_at_y,age_group,months_registered,patient_type
0,28,1,Amoxicillin,2023-03-26,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.830306,-1,NaN,1,John Hill,89.0,Male,2023-08-23,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.814164,71+,20,Regular
1,53,1,Aspirin,2023-03-12,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.830306,-1,NaN,1,John Hill,89.0,Male,2023-08-23,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.814164,71+,20,Regular
2,44,1,Ibuprofen,2023-01-31,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.830306,-1,NaN,1,John Hill,89.0,Male,2023-08-23,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.814164,71+,20,Regular
3,97,1,Metformin,2023-03-06,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.830306,-1,1.0,3,John Hill,89.0,Male,2023-08-23,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.814164,71+,20,Regular
4,121,1,Metformin,2023-03-07,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.830306,1,1.0,3,John Hill,89.0,Male,2023-08-23,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.814164,71+,20,Regular
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,12,45,Metformin,2023-02-17,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.830306,-1,NaN,1,Daniel Barajas,18.0,Male,2023-08-05,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.814164,0-18,20,Regular
124,36,48,Ibuprofen,2023-02-07,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.830306,-1,NaN,1,Lawrence Young,82.0,Male,2024-03-03,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.814164,71+,13,Regular
125,7,49,Amoxicillin,2023-03-04,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.830306,-1,NaN,1,John Ray,61.0,Male,2024-07-19,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.814164,51-70,9,Regular
126,43,49,Atorvastatin,2023-01-13,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.830306,-1,NaN,1,John Ray,61.0,Male,2024-07-19,2025-04-28 01:17:49.722427,2025-04-28 13:00:14.814164,51-70,9,Regular


In [ ]:

df2 = pd.merge(
    df_prescriptions,
    df_patients,
    on='patient_id',
    how='left'
)

grouped = (
    df2.groupby(["age_group", "medication_category"])
    .size()
    .reset_index(name="count")
)


most_common_per_age_group = (
    grouped.sort_values(["age_group", "count"], ascending=[True, False])
    .drop_duplicates(subset=["age_group"])
    .reset_index(drop=True)
)

print(most_common_per_age_group)


  age_group medication_category  count
0      0-18               Other      2
1     19-30               Other      5
2     31-50               Other     16
3     51-70               Other     19
4       71+               Other     25


In [ ]:
# How does prescription frequency correlate with appointment frequency?
patient_appointments = df1.groupby('patient_id').size().reset_index(name='num_appointments')

# 2. Número de prescrições por paciente
patient_prescriptions = df_prescriptions.groupby('patient_id').size().reset_index(name='num_prescriptions')

# 3. Juntar as duas tabelas
patient_activity = patient_appointments.merge(
    patient_prescriptions,
    on='patient_id',
    how='inner'  
)

correlation = patient_activity['num_appointments'].corr(patient_activity['num_prescriptions'])
print(f"Correlação entre número de consultas e número de prescrições: {correlation:.2f}")



Correlação entre número de consultas e número de prescrições: -0.13
